#### LangGraph

- 워크 플로 프레임워크
- LCEL 선형 (A->B->C) / LangGraph 순환 (A-B-A-C-B) 같은 흐름 지원
- 개념
    - node : 실행 할 함수
    - edge : 노드 간 연결
    - stage: 전체 상태를 담는 딕셔너리
- 조건부 엣지, Agent 루프, 자기 수정, 멀티 에이전트 등 복잡한 패턴 구현

In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

### TypeDict / Pydantic
- typeddict : 딕셔너리 키, 값 타입을 정의 할 수 있게 도와줌, 단순 State 구현 시 사용, 기본값 줄 수 없음
- Pydantic : validation 검사

In [2]:
# 데이터 저장소 생성

class MyState(TypedDict):
    message: str

def say_hello(state):
    return {"message" : "Hello, LangGraph!"}

# 그래프 생성
graph = StateGraph(MyState)
graph.add_node("hello", say_hello)

graph.add_edge(START, "hello")
graph.add_edge("hello", END)


# 그래프 실행
app = graph.compile()
result = app.invoke({"message" : ""})
print(result)

# 그래프 시각화 (참고)
app.get_graph().print_ascii()

{'message': 'Hello, LangGraph!'}
+-----------+  
| __start__ |  
+-----------+  
      *        
      *        
      *        
  +-------+    
  | hello |    
  +-------+    
      *        
      *        
      *        
 +---------+   
 | __end__ |   
 +---------+   


### State
- 상태를 중심으로 동작
- TypeDict, Pydantic Base Model을 사용하여 정의
- 그래프 실행 중 지속적으로 업데이트 됨
- 노드간의 전환은 조건부 엣지를 통해 제어 가능하며 복잡한 의사결정 프로세스 모델링 가능
- 재귀적 실행 지원

### Node
- 실제 작업을 수행하는 기본단위
- 함수 기반
- 상태 중심 : 현재 상태를 입력으로 받아 처리
- 독립적 실행 : 각 노드는 독립적으로 실행
- 조합 가능 : 여러 노드를 연결하여 복잡한 워크 플로 가능

### Edge

- 노드 간의 연결과 실행 흐름을 정의

In [5]:
# 카운터 공유
class CounterState(TypedDict):
    count: int

# 증가 함수
def increment(state):
    print(f"현재 카운트 : {state['count']}")
    new_count = state['count'] + 1
    print(f"새로운 카운트 : {new_count}")

    # state 값 변경 (return)
    return {"count" : new_count}

# 그래프 생성
graph = StateGraph(CounterState)
graph.add_node("increment", increment)

# 그래프 연결 (그래프가 실행될 것인가)
graph.add_edge(START, "increment")
graph.add_edge("increment", END)

# 그래프를 실행 가능한 형태로 변경
app = graph.compile()
result = app.invoke({"count":0})
print(f"최종 결과 {result}")

app.get_graph().print_ascii()

현재 카운트 : 0
새로운 카운트 : 1
최종 결과 {'count': 1}
+-----------+  
| __start__ |  
+-----------+  
      *        
      *        
      *        
+-----------+  
| increment |  
+-----------+  
      *        
      *        
      *        
 +---------+   
 | __end__ |   
 +---------+   


In [ ]:
class CounterState(TypedDict):
    count: int


graph = StateGraph(CounterState)

# 2개의 노드


def first_increment(state):
    # +1
    return {"count": state["count"] + 1}


def second_increment(state):
    # +10
    return {"count": state["count"] + 10}


# START => first => second => END
graph.add_node("increment1", first_increment)
graph.add_node("increment2", second_increment)

graph.add_edge(START, "increment1")
graph.add_edge("increment1", "increment2")
graph.add_edge("increment2", END)

app = graph.compile()
result = app.invoke({"count" : 0})
print(f"최종 결과 {result}")
app.get_graph().print_ascii()

최종 결과 {'count': 11}
+-----------+  
| __start__ |  
+-----------+  
       *       
       *       
       *       
+------------+ 
| increment1 | 
+------------+ 
       *       
       *       
       *       
+------------+ 
| increment2 | 
+------------+ 
       *       
       *       
       *       
  +---------+  
  | __end__ |  
  +---------+  


### 조건부 엣지
 - 런타임 상태에 따라 동적으로 실행 경로 결정
 - add_conditional_edges()

In [16]:
# 입력 숫자 > 10  일때 => big
# 입력 숫자 < 10  일때 => small

# State : 숫자, 결과
class NumberState(TypedDict):
    number : int
    result : str

# 노드 (result 값 변경)
def handle_big_number(state):
    return {'result' : f"{state['number']} 는 큰 숫자 입니다."}

def handle_small_number(state):
    return {"result": f"{state['number']} 는 작은 숫자 입니다."}

# 그래프 생성
graph = StateGraph(NumberState)
graph.add_node("big", handle_big_number)
graph.add_node("small", handle_small_number)

# 라우터 (조건 함수)
def check_size(state):
    if state['number'] > 10:
        return "big"
    else:
        return "small"

# 그래프 생성
graph = StateGraph(NumberState)
graph.add_node("big_handler", handle_big_number)
graph.add_node("small_handler", handle_small_number)

# 엣지
graph.add_edge("big_handler", END)
graph.add_edge("small_handler", END)

# 조건부 엣지
graph.add_conditional_edges(START, check_size, {"big" : "big_handler", "small" : "small_handler" })

app = graph.compile()
result = app.invoke({"number" : 15, "result" : ""})
print(f"큰 숫자 {result}")

result = app.invoke({"number": 5, "result": ""})
print(f"작은 숫자 {result}")

app.get_graph().print_ascii()

큰 숫자 {'number': 15, 'result': '15 는 큰 숫자 입니다.'}
작은 숫자 {'number': 5, 'result': '5 는 작은 숫자 입니다.'}
              +-----------+                 
              | __start__ |                 
              +-----------+                 
              ..           ..               
            ..               ..             
          ..                   ..           
+-------------+           +---------------+ 
| big_handler |           | small_handler | 
+-------------+           +---------------+ 
              **           **               
                **       **                 
                  **   **                   
                +---------+                 
                | __end__ |                 
                +---------+                 
